In [1]:


assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing in the Pi container."
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is missing in the Pi container."
assert CASES_PATH.exists(), f"Benchmark file not found: {CASES_PATH}"

with CASES_PATH.open(encoding="utf-8") as file:
    CASES = json.load(file)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

node_version = subprocess.run(
    ["node", "--version"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("Pi notebook setup: OK")
print("Node.js:", node_version)
print("OpenAI API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Tavily API key loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("Model:", MODEL)
print("Benchmark cases:", list(CASES.keys()))
print("Results folder:", RESULTS_DIR)

Pi notebook setup: OK
Node.js: v22.23.3
OpenAI API key loaded: True
Tavily API key loaded: True
Model: gpt-6-luna
Benchmark cases: ['long_prompt_math', 'web_research_ptt']
Results folder: /workspace/shared/results


In [2]:
js_smoke_test = r"""
import process from "node:process";
import {
  createAgentSession,
  DefaultResourceLoader,
  getAgentDir,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";

if (!process.env.OPENAI_API_KEY) {
  throw new Error("OPENAI_API_KEY is missing");
}

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) {
  throw new Error("Pi could not find openai/gpt-6-luna");
}

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace",
  agentDir: getAgentDir(),
  systemPromptOverride: () =>
    "Reply with exactly: PI_AGENT_SDK_OK",
  appendSystemPromptOverride: () => [],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
  noTools: true,
  thinkingLevel: "medium",
});

try {
  await session.prompt("Run the smoke test.");
  console.log("Pi response:", session.getLastAssistantText() ?? "(empty)");
  console.log("Model:", `${model.provider}/${model.id}`);
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--input-type=module", "-e", js_smoke_test],
    cwd=str(PI_DIR),
    capture_output=True,
    text=True,
)

print(run.stdout)

if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f"Pi smoke test failed with exit code {run.returncode}")

Pi response: I’ll locate and run the repository’s smoke test.PI_AGENT_SDK_OK
Model: openai/gpt-6-luna



In [5]:
js_math_benchmark = r"""
import fs from "node:fs/promises";
import process from "node:process";
import { performance } from "node:perf_hooks";
import {
  createAgentSession,
  DefaultResourceLoader,
  getAgentDir,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";

const cases = JSON.parse(
  await fs.readFile("/workspace/shared/benchmark_cases.json", "utf8")
);
const benchmark = cases.long_prompt_math;

if (!benchmark) {
  throw new Error("Benchmark case 'long_prompt_math' was not found");
}
if (!process.env.OPENAI_API_KEY) {
  throw new Error("OPENAI_API_KEY is missing");
}

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) {
  throw new Error("Pi could not find openai/gpt-6-luna");
}

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace",
  agentDir: getAgentDir(),
  systemPromptOverride: () =>
    "Follow every requirement in the user request. Do not use tools.",
  appendSystemPromptOverride: () => [],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
  noTools: true,
  thinkingLevel: "medium",
});

try {
  const startedAt = performance.now();
  await session.prompt(benchmark.prompt);
  const latencyS = (performance.now() - startedAt) / 1000;

  const answer = session.getLastAssistantText() ?? "";
  const assistantMessages = session.messages.filter(
    (message) => message.role === "assistant"
  );
  const usages = assistantMessages
    .map((message) => message.usage)
    .filter(Boolean);

  const usage = usages.reduce(
    (total, item) => ({
      requests: total.requests + 1,
      input_tokens: total.input_tokens + item.input,
      output_tokens: total.output_tokens + item.output,
      reasoning_tokens: total.reasoning_tokens + (item.reasoning ?? 0),
      total_tokens: total.total_tokens + item.totalTokens,
    }),
    {
      requests: 0,
      input_tokens: 0,
      output_tokens: 0,
      reasoning_tokens: 0,
      total_tokens: 0,
    }
  );

  const runId =
    "pi-long-math-" +
    new Date().toISOString().replace(/[-:]/g, "").replace(/\.\d{3}Z$/, "Z");

  const record = {
    sdk: "pi-agent-sdk",
    case: "long_prompt_math",
    run_id: runId,
    model: `${model.provider}/${model.id}`,
    thinking_level: "medium",
    tools_enabled: false,
    latency_s: Number(latencyS.toFixed(2)),
    usage,
    requirements: benchmark.requirements,
    run_item_types: [...new Set(session.messages.map((message) => message.role))],
    answer,
  };

  const resultsDir = "/workspace/shared/results";
  await fs.mkdir(resultsDir, { recursive: true });

  const outputPath = `${resultsDir}/${runId}.json`;
  await fs.writeFile(outputPath, JSON.stringify(record, null, 2), "utf8");

  console.log("Answer:\n" + answer);
  console.log("\nRun ID:", runId);
  console.log("Latency (seconds):", record.latency_s);
  console.log("Usage:", JSON.stringify(usage));
  console.log("Saved:", outputPath);
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--input-type=module", "-e", js_math_benchmark],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
)

print(run.stdout)

if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f"Pi math benchmark failed with exit code {run.returncode}")

Answer:
## Executive summary

Using only the **supplied test data** (not externally verified facts), revenue grew **20%** and net income grew **27.5%** from FY2024 to FY2025. Net margin increased from **8.0%** to **8.5%**. FY2024 net debt was **THB 200 million**. FY2025 net debt is **unavailable** because cash was not reported.

## Financial table

All monetary figures are **supplied test data** in million THB; calculated percentages and net debt are derived from that test data.

| Metric | FY2024 | FY2025 |
|---|---:|---:|
| Revenue | 1,000 | 1,200 |
| Net income | 80 | 102 |
| Total debt | 300 | 280 |
| Cash | 100 | Not reported |
| Revenue growth from FY2024 | — | 20% |
| Net income growth from FY2024 | — | 27.5% |
| Net margin | 8.0% | 8.5% |
| Net debt | 200 | Unavailable |

## Calculation notes

- **Revenue growth:**  
  \((\text{FY2025 revenue} - \text{FY2024 revenue}) \div \text{FY2024 revenue} \times 100\)  
  \(= (1{,}200 - 1{,}000) \div 1{,}000 \times 100 = 20\%\)

- **Net i

In [8]:
import json
import os
import statistics
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

PROJECT_DIR = Path("/workspace")
PI_DIR = PROJECT_DIR / "pi_agent"
SHARED_DIR = PROJECT_DIR / "shared"
RESULTS_DIR = SHARED_DIR / "results"
CASES_PATH = SHARED_DIR / "benchmark_cases.json"

MODEL = "gpt-6-luna"

In [13]:
# js_mcp_test = r"""
# import fs from "node:fs/promises";
# import process from "node:process";
# import {
#   createAgentSession,
#   DefaultResourceLoader,
#   getAgentDir,
#   ModelRuntime,
#   SessionManager,
# } from "@earendil-works/pi-coding-agent";
# import { createMcpAdapter } from "pi-mcp-adapter";

# if (!process.env.OPENAI_API_KEY) {
#   throw new Error("OPENAI_API_KEY is missing");
# }
# if (!process.env.TAVILY_API_KEY) {
#   throw new Error("TAVILY_API_KEY is missing");
# }

# const adapterPackage = JSON.parse(
#   await fs.readFile("/workspace/node_modules/pi-mcp-adapter/package.json", "utf8")
# );

# const modelRuntime = await ModelRuntime.create();
# await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

# const model = modelRuntime.getModel("openai", "gpt-6-luna");
# if (!model) {
#   throw new Error("Pi could not find openai/gpt-6-luna");
# }

# const adapter = createMcpAdapter({
#   config: {
#     mcpServers: {
#       tavily: {
#         url: "https://mcp.tavily.com/mcp/",
#         headers: {
#           Authorization: `Bearer ${process.env.TAVILY_API_KEY}`,
#         },
#         lifecycle: "eager",
#         directTools: true,
#       },
#     },
#   },
# });

# const extensionFactory =
#   typeof adapter === "function" ? adapter : adapter?.extensionFactory;

# if (typeof extensionFactory !== "function") {
#   throw new Error(
#     `Unexpected pi-mcp-adapter result: ${JSON.stringify(Object.keys(adapter ?? {}))}`
#   );
# }

# const resourceLoader = new DefaultResourceLoader({
#   cwd: "/workspace",
#   agentDir: getAgentDir(),
#   extensionFactories: [extensionFactory],
# });

# await resourceLoader.reload();

# const { session } = await createAgentSession({
#   model,
#   modelRuntime,
#   sessionManager: SessionManager.inMemory(),
#   resourceLoader,
# });

# try {
#   await session.bindExtensions({});

#   const activeTools = session.getActiveToolNames();
#   const tavilyTools = activeTools.filter(
#     (name) => name.toLowerCase().includes("tavily")
#   );

#   console.log("pi-mcp-adapter version:", adapterPackage.version);
#   console.log("All active tools:", activeTools);
#   console.log("Tavily direct tools:", tavilyTools);
#   console.log("MCP proxy available:", activeTools.includes("mcp"));

#   if (!activeTools.includes("mcp") && tavilyTools.length === 0) {
#     throw new Error("No MCP proxy or Tavily direct tools are active");
#   }
# } finally {
#   session.dispose();
# }
# """

# run = subprocess.run(
#     ["node", "--import", "tsx", "--input-type=module", "-e", js_mcp_test],
#     cwd=str(PROJECT_DIR),
#     capture_output=True,
#     text=True,
# )

# print(run.stdout)

# if run.returncode != 0:
#     print(run.stderr)
#     raise RuntimeError(f"Pi Tavily MCP test failed with exit code {run.returncode}")

pi-mcp-adapter version: 3.2.0
All active tools: [ 'read', 'bash', 'edit', 'write', 'mcpScript', 'mcp' ]
Tavily direct tools: []
MCP proxy available: true



In [15]:
js_mcp_test = r"""
import fs from "node:fs/promises";
import process from "node:process";
import {
  createAgentSession,
  DefaultResourceLoader,
  getAgentDir,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";
import { createMcpAdapter } from "pi-mcp-adapter";

if (!process.env.OPENAI_API_KEY) {
  throw new Error("OPENAI_API_KEY is missing");
}
if (!process.env.TAVILY_API_KEY) {
  throw new Error("TAVILY_API_KEY is missing");
}

const adapterPackage = JSON.parse(
  await fs.readFile("/workspace/node_modules/pi-mcp-adapter/package.json", "utf8")
);

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) {
  throw new Error("Pi could not find openai/gpt-6-luna");
}

const adapter = createMcpAdapter({
  config: {
    mcpServers: {
      tavily: {
        url: "https://mcp.tavily.com/mcp/",
        headers: {
          Authorization: `Bearer ${process.env.TAVILY_API_KEY}`,
        },
        lifecycle: "eager",
        directTools: true,
      },
    },
  },
});

const extensionFactory =
  typeof adapter === "function" ? adapter : adapter?.extensionFactory;

if (typeof extensionFactory !== "function") {
  throw new Error(
    `Unexpected pi-mcp-adapter result: ${JSON.stringify(Object.keys(adapter ?? {}))}`
  );
}

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace",
  agentDir: getAgentDir(),
  extensionFactories: [extensionFactory],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
});

try {
  await session.bindExtensions({});

  const activeTools = session.getActiveToolNames();
  const tavilyTools = activeTools.filter(
    (name) => name.toLowerCase().includes("tavily")
  );

  console.log("pi-mcp-adapter version:", adapterPackage.version);
  console.log("All active tools:", activeTools);
  console.log("Tavily direct tools:", tavilyTools);
  console.log("MCP proxy available:", activeTools.includes("mcp"));

  if (!activeTools.includes("mcp") && tavilyTools.length === 0) {
    throw new Error("No MCP proxy or Tavily direct tools are active");
  }

  await session.prompt(
    "Use only the mcp tool. Connect to the MCP server named tavily, then search for the official PTT Public Company Limited investor-relations homepage. Return one result's title and URL from the tool output. If the MCP call fails, report that instead of guessing."
  );

  console.log(
    "Pi response:",
    session.getLastAssistantText() ?? "(empty)"
  );

  const toolCalls = session.messages
    .flatMap((message) =>
      Array.isArray(message.content) ? message.content : []
    )
    .filter((item) => item.type === "toolCall")
    .map((item) => item.name);

  console.log("Tool calls:", toolCalls);

  const usedMcp = toolCalls.some(
    (name) => name === "mcp" || name.toLowerCase().includes("tavily")
  );

  if (!usedMcp) {
    throw new Error("Pi did not call the Tavily MCP tool");
  }
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--import", "tsx", "--input-type=module", "-e", js_mcp_test],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
)

print(run.stdout)

if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f"Pi Tavily MCP test failed with exit code {run.returncode}")

pi-mcp-adapter version: 3.2.0
All active tools: [
  'read',
  'bash',
  'edit',
  'write',
  'mcpScript',
  'tavily_search',
  'tavily_extract',
  'tavily_crawl',
  'tavily_map',
  'tavily_research',
  'mcp'
]
Tavily direct tools: [
  'tavily_search',
  'tavily_extract',
  'tavily_crawl',
  'tavily_map',
  'tavily_research'
]
MCP proxy available: true
Pi response: Title: PTT Public Company Limited (PTT)  
URL: https://investor.pttplc.com/en/ir-home
Tool calls: [ 'mcp', 'mcp' ]



In [16]:
import subprocess

js_ptt_benchmark = r"""
import fs from "node:fs/promises";
import process from "node:process";
import { performance } from "node:perf_hooks";
import {
  createAgentSession,
  DefaultResourceLoader,
  getAgentDir,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";
import { createMcpAdapter } from "pi-mcp-adapter";

const failIfMissing = (name) => {
  if (!process.env[name]) throw new Error(`${name} is missing`);
};
failIfMissing("OPENAI_API_KEY");
failIfMissing("TAVILY_API_KEY");

const cases = JSON.parse(
  await fs.readFile("/workspace/shared/benchmark_cases.json", "utf8")
);
const benchmark = cases.web_research_ptt;
if (!benchmark) throw new Error("Benchmark case 'web_research_ptt' was not found");

const runId =
  "pi-ptt-research-" +
  new Date().toISOString().replace(/[-:]/g, "").replace(/\.\d{3}Z$/, "Z");

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) throw new Error("Pi could not find openai/gpt-6-luna");

const adapter = createMcpAdapter({
  config: {
    mcpServers: {
      tavily: {
        url: "https://mcp.tavily.com/mcp/",
        headers: {
          Authorization: `Bearer ${process.env.TAVILY_API_KEY}`,
        },
        lifecycle: "eager",
        directTools: true,
      },
    },
  },
});

const extensionFactory =
  typeof adapter === "function" ? adapter : adapter?.extensionFactory;
if (typeof extensionFactory !== "function") {
  throw new Error("Could not get the Pi MCP adapter extension factory");
}

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace",
  agentDir: getAgentDir(),
  extensionFactories: [extensionFactory],
  systemPromptOverride: () =>
    "Use Tavily MCP search and retrieval tools to answer the user's request. " +
    "Prioritize official PTT and SET sources. Search efficiently and stop " +
    "once sources support all requested facts. Include source URLs for factual " +
    "claims. Distinguish reported figures from calculated figures. Do not guess.",
  appendSystemPromptOverride: () => [],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
  noTools: "builtin",
  thinkingLevel: "medium",
});

let status = "completed";
let error = null;
let latencyS = 0;

try {
  await session.bindExtensions({});

  const activeTools = session.getActiveToolNames();
  if (!activeTools.includes("mcp") &&
      !activeTools.some((name) => name.startsWith("tavily_"))) {
    throw new Error(`No Tavily MCP tools are active. Active tools: ${activeTools}`);
  }

  const startedAt = performance.now();
  try {
    await session.prompt(benchmark.prompt);
  } catch (err) {
    status = "failed";
    error = String(err);
  }
  latencyS = (performance.now() - startedAt) / 1000;

  const answer = session.getLastAssistantText() ?? "";
  const assistantMessages = session.messages.filter(
    (message) => message.role === "assistant"
  );
  const usage = assistantMessages
    .map((message) => message.usage)
    .filter(Boolean)
    .reduce(
      (total, item) => ({
        requests: total.requests + 1,
        input_tokens: total.input_tokens + (item.input ?? 0),
        output_tokens: total.output_tokens + (item.output ?? 0),
        reasoning_tokens: total.reasoning_tokens + (item.reasoning ?? 0),
        total_tokens: total.total_tokens + (item.totalTokens ?? 0),
      }),
      {
        requests: 0,
        input_tokens: 0,
        output_tokens: 0,
        reasoning_tokens: 0,
        total_tokens: 0,
      }
    );

  const toolCalls = session.messages
    .flatMap((message) => Array.isArray(message.content) ? message.content : [])
    .filter((item) => item.type === "toolCall")
    .map((item) => item.name);

  const record = {
    sdk: "pi-agent-sdk+pi-mcp-adapter",
    case: "web_research_ptt",
    run_id: runId,
    model: `${model.provider}/${model.id}`,
    mcp_adapter: "pi-mcp-adapter",
    mcp_server: "tavily",
    thinking_level: "medium",
    no_builtin_tools: true,
    status,
    latency_s: Number(latencyS.toFixed(2)),
    usage,
    tool_calls: toolCalls,
    active_tools: session.getActiveToolNames(),
    requirements: benchmark.requirements,
    answer,
    ...(error ? { error } : {}),
  };

  const resultsDir = "/workspace/shared/results";
  await fs.mkdir(resultsDir, { recursive: true });
  const outputPath = `${resultsDir}/${runId}.json`;
  await fs.writeFile(outputPath, JSON.stringify(record, null, 2), "utf8");

  console.log("Status:", status);
  console.log("Answer:\n" + (answer || "(No answer returned)"));
  console.log("Run ID:", runId);
  console.log("Latency (seconds):", record.latency_s);
  console.log("Usage:", JSON.stringify(usage));
  console.log("Tool calls:", JSON.stringify(toolCalls));
  if (error) console.log("Error:", error);
  console.log("Saved:", outputPath);
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--import", "tsx", "--input-type=module", "-e", js_ptt_benchmark],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print(run.stdout)
if run.stderr:
    print("STDERR:\n", run.stderr)
if run.returncode != 0:
    raise RuntimeError(f"Pi PTT research benchmark failed: exit code {run.returncode}")

Status: completed
Answer:
## PTT Public Company Limited — as of 29 September 2026

### Identity and latest completed fiscal year
The Stock Exchange of Thailand identifies **PTT** as the ticker for **PTT Public Company Limited**. [SET company profile](https://www.set.or.th/en/market/product/stock/quote/PTT/company-profile/information)

PTT’s most recently published completed fiscal year was **FY2025, ended 31 December 2025**. PTT’s investor-relations site lists its 2025 annual report and FY2025 financial statements; its annual MD&A covers the year ended 31 December 2025. [PTT annual-report downloads](https://investor.pttplc.com/en/downloads/one-report) · [PTT FY2025 MD&A](https://core.shareinvestor.app/storage/downloads/ptt/mdna/0646NWS190220261820230740E.pdf)

### FY2025 financial results
Both figures are **reported**, not calculated, and are for PTT and its subsidiaries:

| Measure | Reported amount | Period and description |
|---|---:|---|
| Revenue | **THB 2,662,145 million** | FY20

In [19]:
import json
from pathlib import Path

path = Path(
    "/workspace/shared/results/pi-ptt-research-20260929T105524Z.json"
)

print("File exists:", path.exists())
print("Path:", path)

if not path.exists():
    results_dir = Path("/workspace/shared/results")
    print("Available result files:")
    print([p.name for p in results_dir.glob("*.json")][-10:])
else:
    record = json.loads(path.read_text(encoding="utf-8"))
    usage = record["usage"]

    print("Status:", record.get("status"))
    print("Usage:", json.dumps(usage, indent=2))
    print("Input + output:", usage["input_tokens"] + usage["output_tokens"])
    print("Reported total:", usage["total_tokens"])
    print("Tool calls:", len(record.get("tool_calls", [])))

File exists: True
Path: /workspace/shared/results/pi-ptt-research-20260929T105524Z.json
Status: completed
Usage: {
  "requests": 14,
  "input_tokens": 42,
  "output_tokens": 4756,
  "reasoning_tokens": 3133,
  "total_tokens": 635168
}
Input + output: 4798
Reported total: 635168
Tool calls: 13


In [21]:
import subprocess

js_ptt_benchmark = r"""
import fs from "node:fs/promises";
import process from "node:process";
import { performance } from "node:perf_hooks";
import {
  createAgentSession,
  DefaultResourceLoader,
  getAgentDir,
  ModelRuntime,
  SessionManager,
} from "@earendil-works/pi-coding-agent";
import { createMcpAdapter } from "pi-mcp-adapter";

const failIfMissing = (name) => {
  if (!process.env[name]) throw new Error(`${name} is missing`);
};
failIfMissing("OPENAI_API_KEY");
failIfMissing("TAVILY_API_KEY");

const cases = JSON.parse(
  await fs.readFile("/workspace/shared/benchmark_cases.json", "utf8")
);
const benchmark = cases.web_research_ptt;
if (!benchmark) {
  throw new Error("Benchmark case 'web_research_ptt' was not found");
}

const runId =
  "pi-ptt-research-" +
  new Date().toISOString().replace(/[-:]/g, "").replace(/\.\d{3}Z$/, "Z");

const modelRuntime = await ModelRuntime.create();
await modelRuntime.setRuntimeApiKey("openai", process.env.OPENAI_API_KEY);

const model = modelRuntime.getModel("openai", "gpt-6-luna");
if (!model) throw new Error("Pi could not find openai/gpt-6-luna");

const adapter = createMcpAdapter({
  config: {
    mcpServers: {
      tavily: {
        url: "https://mcp.tavily.com/mcp/",
        headers: {
          Authorization: `Bearer ${process.env.TAVILY_API_KEY}`,
        },
        lifecycle: "eager",
        directTools: true,
      },
    },
  },
});

const extensionFactory =
  typeof adapter === "function" ? adapter : adapter?.extensionFactory;
if (typeof extensionFactory !== "function") {
  throw new Error("Could not get the Pi MCP adapter extension factory");
}

const resourceLoader = new DefaultResourceLoader({
  cwd: "/workspace",
  agentDir: getAgentDir(),
  extensionFactories: [extensionFactory],
  systemPromptOverride: () =>
    "Use Tavily MCP search and retrieval tools to answer the user's request. " +
    "Prioritize official PTT and SET sources. Search efficiently and stop " +
    "once sources support all requested facts. Include source URLs for factual " +
    "claims. Distinguish reported figures from calculated figures. Do not guess.",
  appendSystemPromptOverride: () => [],
});

await resourceLoader.reload();

const { session } = await createAgentSession({
  model,
  modelRuntime,
  sessionManager: SessionManager.inMemory(),
  resourceLoader,
  noTools: "builtin",
  thinkingLevel: "medium",
});

let status = "completed";
let error = null;
let latencyS = 0;

try {
  await session.bindExtensions({});

  const activeTools = session.getActiveToolNames();
  if (
    !activeTools.includes("mcp") &&
    !activeTools.some((name) => name.startsWith("tavily_"))
  ) {
    throw new Error(`No Tavily MCP tools are active. Active tools: ${activeTools}`);
  }

  const startedAt = performance.now();
  try {
    await session.prompt(benchmark.prompt);
  } catch (err) {
    status = "failed";
    error = String(err);
  }
  latencyS = (performance.now() - startedAt) / 1000;

  const answer = session.getLastAssistantText() ?? "";
  const assistantMessages = session.messages.filter(
    (message) => message.role === "assistant"
  );
  const usages = assistantMessages
    .map((message) => message.usage)
    .filter(Boolean);

  const usage = usages.reduce(
    (total, item) => ({
      requests: total.requests + 1,
      input_tokens: total.input_tokens + (item.input ?? 0),
      output_tokens: total.output_tokens + (item.output ?? 0),
      cache_read_tokens: total.cache_read_tokens + (item.cacheRead ?? 0),
      cache_write_tokens: total.cache_write_tokens + (item.cacheWrite ?? 0),
      reasoning_tokens: total.reasoning_tokens + (item.reasoning ?? 0),
      total_tokens: total.total_tokens + (item.totalTokens ?? 0),
    }),
    {
      requests: 0,
      input_tokens: 0,
      output_tokens: 0,
      cache_read_tokens: 0,
      cache_write_tokens: 0,
      reasoning_tokens: 0,
      total_tokens: 0,
    }
  );

  usage.input_context_tokens =
    usage.input_tokens +
    usage.cache_read_tokens +
    usage.cache_write_tokens;

  usage.breakdown_total_tokens =
    usage.input_context_tokens + usage.output_tokens;

  usage.total_matches_breakdown =
    usage.total_tokens === usage.breakdown_total_tokens;

  const toolCalls = session.messages
    .flatMap((message) =>
      Array.isArray(message.content) ? message.content : []
    )
    .filter((item) => item.type === "toolCall")
    .map((item) => item.name);

  const record = {
    sdk: "pi-agent-sdk+pi-mcp-adapter",
    case: "web_research_ptt",
    run_id: runId,
    model: `${model.provider}/${model.id}`,
    mcp_adapter: "pi-mcp-adapter",
    mcp_server: "tavily",
    thinking_level: "medium",
    no_builtin_tools: true,
    status,
    latency_s: Number(latencyS.toFixed(2)),
    usage,
    tool_calls: toolCalls,
    active_tools: session.getActiveToolNames(),
    requirements: benchmark.requirements,
    answer,
    ...(error ? { error } : {}),
  };

  const resultsDir = "/workspace/shared/results";
  await fs.mkdir(resultsDir, { recursive: true });

  const outputPath = `${resultsDir}/${runId}.json`;
  await fs.writeFile(outputPath, JSON.stringify(record, null, 2), "utf8");

  console.log("Status:", status);
  console.log("Answer:\n" + (answer || "(No answer returned)"));
  console.log("Run ID:", runId);
  console.log("Latency (seconds):", record.latency_s);
  console.log("Usage:", JSON.stringify(usage));
  console.log("Tool calls:", JSON.stringify(toolCalls));
  if (error) console.log("Error:", error);
  console.log("Saved:", outputPath);
} finally {
  session.dispose();
}
"""

run = subprocess.run(
    ["node", "--import", "tsx", "--input-type=module", "-e", js_ptt_benchmark],
    cwd="/workspace",
    capture_output=True,
    text=True,
)

print(run.stdout)
if run.stderr:
    print("STDERR:\n", run.stderr)
if run.returncode != 0:
    raise RuntimeError(
        f"Pi PTT research benchmark failed: exit code {run.returncode}"
    )

Status: completed
Answer:
(No answer returned)
Run ID: pi-ptt-research-20260929T110822Z
Latency (seconds): 101.77
Usage: {"requests":10,"input_tokens":27,"output_tokens":2880,"cache_read_tokens":231524,"cache_write_tokens":107364,"reasoning_tokens":1053,"total_tokens":341795,"input_context_tokens":338915,"breakdown_total_tokens":341795,"total_matches_breakdown":true}
Tool calls: ["mcpScript","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp","mcp"]
Saved: /workspace/shared/results/pi-ptt-research-20260929T110822Z.json



In [22]:
import json
from pathlib import Path

path = Path(
    "/workspace/shared/results/pi-ptt-research-20260929T110822Z.json"
)

record = json.loads(path.read_text(encoding="utf-8"))

# แยกผลการทำงานของ benchmark ออกจากการจบของโปรแกรม
if not (record.get("answer") or "").strip():
    record["benchmark_status"] = "no_final_answer"
    record["benchmark_note"] = (
        "The run completed without a runtime exception, "
        "but returned no final answer."
    )
else:
    record["benchmark_status"] = "answer_returned"

path.write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Updated:", path)
print("Runtime status:", record.get("status"))
print("Benchmark status:", record.get("benchmark_status"))

Updated: /workspace/shared/results/pi-ptt-research-20260929T110822Z.json
Runtime status: completed
Benchmark status: no_final_answer
